In [ ]:
# Byte-Pair Encoding (BPE) tokenizer from scratch - train, encode, decode.
# Byte-level (GPT-2/GPT-4 style): the corpus is first turned into raw UTF-8 bytes, so
# the base vocabulary is the 256 byte values and ANY input is representable - there is
# no out-of-vocabulary case. Training then greedily merges the most frequent adjacent
# pair, repeatedly, until the vocab reaches the target size; the ordered merge rules ARE
# the model. Encoding replays those merges by learned rank; decoding maps ids back to
# bytes and UTF-8 decodes. See Notes/Note_4_transformer_core.md §6.
# Pure standard library, no dependencies. Reference: github.com/karpathy/minbpe.

In [ ]:
from __future__ import annotations


def get_stats(ids: list[int], counts: dict | None = None) -> dict:
    """Count every adjacent pair in a sequence of token ids: {(a, b): frequency}."""
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids: list[int], pair: tuple[int, int], idx: int) -> list[int]:
    """Replace every occurrence of `pair` in `ids` with the single new token `idx`."""
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(idx)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

In [ ]:
# The tokenizer. `merges` maps each learned pair -> its new token id (the id also encodes
# merge order, since ids are assigned 256, 257, ... in the order pairs are merged).
# `vocab` maps every token id -> the raw bytes it expands to, used for decoding.

class BPETokenizer:
    def __init__(self):
        self.merges: dict[tuple[int, int], int] = {}
        self.vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}

    def train(self, text: str, vocab_size: int, verbose: bool = False) -> None:
        assert vocab_size >= 256, "vocab_size must leave room for the 256 base bytes"
        num_merges = vocab_size - 256
        ids = list(text.encode("utf-8"))               # start from raw bytes
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(256)}
        for k in range(num_merges):
            stats = get_stats(ids)
            if not stats:
                break                                  # corpus fully merged
            pair = max(stats, key=stats.get)           # most frequent adjacent pair
            idx = 256 + k
            ids = merge(ids, pair, idx)
            self.merges[pair] = idx
            self.vocab[idx] = self.vocab[pair[0]] + self.vocab[pair[1]]
            if verbose:
                print(f"merge {k + 1:3d}/{num_merges}: {pair} -> {idx} "
                      f"({self.vocab[idx].decode('utf-8', errors='replace')!r}) "
                      f"[{stats[pair]} occurrences]")

    def encode(self, text: str) -> list[int]:
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = get_stats(ids)
            # Merge the pair learned EARLIEST (lowest id); unknown pairs get +inf.
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break                                  # nothing left to merge
            ids = merge(ids, pair, self.merges[pair])
        return ids

    def decode(self, ids: list[int]) -> str:
        raw = b"".join(self.vocab[i] for i in ids)
        return raw.decode("utf-8", errors="replace")

In [ ]:
# Demo corpus: a short passage with lots of repeated English structure so BPE has
# frequent pairs to discover (' t', 'th', 'the', etc.). Trained to a 320-token vocab.

corpus = (
    "the quick brown fox jumps over the lazy dog. "
    "the dog barks and the fox runs. the quick fox is clever, "
    "the lazy dog is sleepy. these are the themes of the story: "
    "the fox, the dog, and the things they do together. "
) * 20

tok = BPETokenizer()
tok.train(corpus, vocab_size=320, verbose=True)
print(f"\nlearned {len(tok.merges)} merges; total vocab = {len(tok.vocab)}")

In [ ]:
# Verify: round-trip on the training text and on unseen, multilingual + emoji input
# (byte-level => always reversible), and measure how much the merges compress.

def check(label: str, s: str) -> None:
    ids = tok.encode(s)
    back = tok.decode(ids)
    n_bytes = len(s.encode("utf-8"))
    ratio = n_bytes / len(ids) if ids else 0.0
    print(f"{label:14s} bytes={n_bytes:5d} tokens={len(ids):5d} "
          f"compression={ratio:.2f}x  round-trip={'OK' if back == s else 'FAIL'}")
    assert back == s, f"round-trip failed for {label!r}"

check("train text", corpus)
check("unseen en", "the clever fox jumped over another lazy dog today.")
check("unicode", "héllo wörld \u4f60\u597d \U0001f98a\U0001f436")   # accents, CJK, emoji
check("empty", "")

# Show a learned multi-byte token and how a word tokenizes.
word = "the"
ids = tok.encode(word)
print(f"\nencode({word!r}) -> {ids} -> pieces "
      f"{[tok.vocab[i].decode('utf-8', errors='replace') for i in ids]}")
print("all round-trips passed.")